In [ ]:
"""
RQ3 — Lightweight Subgroup Analysis
======================================
Dissertation: Predicting Revenue Growth and Cost Reduction from Business AI Adoption

PURPOSE
-------
Directly answers RQ3: "How do predicted revenue growth and cost reduction
vary across industries, company sizes, and regions?"

This is the LIGHTWEIGHT (Option 1) implementation confirmed in your scope
decisions: NO new models are trained and NO new SHAP analysis is run. This
script simply:
  1. Loads the already-trained XGBoost models (primary model) for both targets
  2. Generates predictions for the full dataset (all 150,000 companies)
  3. Groups those predictions by industry / company_size / region
  4. Produces grouped mean tables + grouped bar charts

This directly reuses the models already saved by shared_pipeline.py —
no retraining, no new SHAP computation, consistent with the "lightweight"
scope decision to keep RQ3 low-cost relative to RQ2/RQ4.

Run: python subgroup_analysis.py
Requires: shared_pipeline.py must have been run (FULL run) for BOTH targets
          first, so that the trained XGBoost models exist in
          pipeline_outputs/models/.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os

# ═══════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════

DATA_PATH = "ai_company_adoption.csv"   # <-- adjust to your actual filename
MODELS_DIR = "pipeline_outputs/models"
OUT_DIR = "pipeline_outputs/subgroup_analysis"
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_1 = "revenue_growth_percent"
TARGET_2 = "cost_reduction_percent"
TARGET_1_LABEL = "Revenue Growth"
TARGET_2_LABEL = "Cost Reduction"

GROUP_VARS = ["industry", "company_size", "region"]  # adjust if your column names differ

NAVY = "#1F4E79"
AMBER = "#E8A33D"

# ═══════════════════════════════════════════════════════════════════════════
# 1. LOAD THE TRAINED XGBOOST MODELS (primary model — no retraining)
# ═══════════════════════════════════════════════════════════════════════════

def load_model(target_column, models_dir=MODELS_DIR):
    path = f"{models_dir}/{target_column}_xgboost.pkl"
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"\nCould not find '{path}'.\n"
            f"Make sure shared_pipeline.py has been run with sample_frac=None "
            f"(a FULL run, not a quick test) for target_column='{target_column}' first, "
            f"so the trained XGBoost model has been saved."
        )
    with open(path, "rb") as f:
        return pickle.load(f)

print("=" * 80)
print("LOADING TRAINED MODELS (reusing existing XGBoost models — no retraining)")
print("=" * 80)

model_1 = load_model(TARGET_1)
model_2 = load_model(TARGET_2)
print(f"✅ Loaded trained XGBoost model for {TARGET_1}")
print(f"✅ Loaded trained XGBoost model for {TARGET_2}")

# ═══════════════════════════════════════════════════════════════════════════
# 2. LOAD THE FULL DATASET AND GENERATE PREDICTIONS
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("LOADING FULL DATASET AND GENERATING PREDICTIONS")
print("=" * 80)

try:
    df = pd.read_csv(DATA_PATH)
    print(f"✅ Dataset loaded: {df.shape[0]:,} rows")
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nCould not find '{DATA_PATH}'.\n"
        f"Update DATA_PATH at the top of this script to point to your CSV file."
    )

missing_group_vars = [g for g in GROUP_VARS if g not in df.columns]
if missing_group_vars:
    print(f"\n⚠ WARNING — these grouping columns were not found: {missing_group_vars}")
    print(f"  Available columns: {list(df.columns)}")
    print(f"  Update GROUP_VARS at the top of this script to match your actual column names.\n")
    GROUP_VARS = [g for g in GROUP_VARS if g in df.columns]

# The trained models are full sklearn Pipelines (preprocessor + model bundled),
# so .predict() can be called directly on a dataframe containing the raw
# (unencoded, unscaled) feature columns exactly as used during training.
# We drop rows with missing values in required features first, matching the
# same cleaning step used during training (Section 3.4/3.5).
feature_cols = [c for c in df.columns if c not in [TARGET_1, TARGET_2]]
df_clean = df.dropna(subset=[c for c in feature_cols if c in df.columns])
print(f"Rows retained after dropping missing values: {len(df_clean):,} (dropped {len(df) - len(df_clean):,})")

X_full = df_clean[[c for c in feature_cols if c in df_clean.columns]]

pred_revenue = model_1.predict(X_full)
pred_cost = model_2.predict(X_full)

results_df = df_clean[GROUP_VARS].copy()
results_df["predicted_" + TARGET_1] = pred_revenue
results_df["predicted_" + TARGET_2] = pred_cost

print(f"\n✅ Predictions generated for {len(results_df):,} companies")
print(f"   Predicted {TARGET_1} range: [{pred_revenue.min():.2f}, {pred_revenue.max():.2f}]")
print(f"   Predicted {TARGET_2} range: [{pred_cost.min():.2f}, {pred_cost.max():.2f}]")

results_df.to_csv(f"{OUT_DIR}/rq3_predictions_with_subgroups.csv", index=False)
print(f"✅ Saved: {OUT_DIR}/rq3_predictions_with_subgroups.csv")

# ═══════════════════════════════════════════════════════════════════════════
# 3. GROUPED MEANS PER SUBGROUP VARIABLE
# ═══════════════════════════════════════════════════════════════════════════

pred_cols = [f"predicted_{TARGET_1}", f"predicted_{TARGET_2}"]
all_summary_tables = {}

for group_var in GROUP_VARS:
    print("\n" + "=" * 80)
    print(f"GROUPED MEANS BY: {group_var}")
    print("=" * 80)

    grouped = results_df.groupby(group_var)[pred_cols].agg(["mean", "std", "count"])
    grouped.columns = ["_".join(col) for col in grouped.columns]
    grouped = grouped.sort_values(f"predicted_{TARGET_1}_mean", ascending=False)

    print(grouped.round(3).to_string())
    grouped.to_csv(f"{OUT_DIR}/rq3_grouped_means_by_{group_var}.csv")
    print(f"\n✅ Saved: {OUT_DIR}/rq3_grouped_means_by_{group_var}.csv")

    all_summary_tables[group_var] = grouped

# ═══════════════════════════════════════════════════════════════════════════
# 4. GROUPED BAR CHARTS — one per subgroup variable
# ═══════════════════════════════════════════════════════════════════════════

for group_var in GROUP_VARS:
    means = results_df.groupby(group_var)[pred_cols].mean()
    means = means.sort_values(f"predicted_{TARGET_1}", ascending=False)

    fig, ax = plt.subplots(figsize=(max(8, len(means) * 1.1), 5.5))
    x = np.arange(len(means))
    width = 0.35

    ax.bar(x - width/2, means[f"predicted_{TARGET_1}"], width, label=TARGET_1_LABEL, color=NAVY, alpha=0.9)
    ax.bar(x + width/2, means[f"predicted_{TARGET_2}"], width, label=TARGET_2_LABEL, color=AMBER, alpha=0.9)

    ax.set_xlabel(group_var.replace("_", " ").title(), fontsize=10)
    ax.set_ylabel("Mean predicted value (%)", fontsize=10)
    ax.set_title(
        f"RQ3 — Predicted {TARGET_1_LABEL} vs {TARGET_2_LABEL}\nby {group_var.replace('_', ' ').title()}",
        fontsize=12, fontweight="bold"
    )
    ax.set_xticks(x)
    ax.set_xticklabels(means.index, rotation=35, ha="right", fontsize=9)
    ax.legend(fontsize=9)
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/rq3_chart_{group_var}.png", dpi=150, bbox_inches="tight")
    print(f"✅ Saved: {OUT_DIR}/rq3_chart_{group_var}.png")
    plt.close()

# ═══════════════════════════════════════════════════════════════════════════
# 5. SUMMARY — top/bottom category per grouping variable (for write-up)
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("SUMMARY — highest/lowest predicted category per grouping variable")
print("=" * 80)

summary_rows = []
for group_var in GROUP_VARS:
    means = results_df.groupby(group_var)[pred_cols].mean()
    top_revenue = means[f"predicted_{TARGET_1}"].idxmax()
    bottom_revenue = means[f"predicted_{TARGET_1}"].idxmin()
    top_cost = means[f"predicted_{TARGET_2}"].idxmax()
    bottom_cost = means[f"predicted_{TARGET_2}"].idxmin()

    summary_rows.append({
        "grouping_variable": group_var,
        f"highest_{TARGET_1}": top_revenue,
        f"highest_{TARGET_1}_value": round(float(means.loc[top_revenue, f"predicted_{TARGET_1}"]), 2),
        f"lowest_{TARGET_1}": bottom_revenue,
        f"lowest_{TARGET_1}_value": round(float(means.loc[bottom_revenue, f"predicted_{TARGET_1}"]), 2),
        f"highest_{TARGET_2}": top_cost,
        f"highest_{TARGET_2}_value": round(float(means.loc[top_cost, f"predicted_{TARGET_2}"]), 2),
        f"lowest_{TARGET_2}": bottom_cost,
        f"lowest_{TARGET_2}_value": round(float(means.loc[bottom_cost, f"predicted_{TARGET_2}"]), 2),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
summary_df.to_csv(f"{OUT_DIR}/rq3_summary_table.csv", index=False)
print(f"\n✅ Saved: {OUT_DIR}/rq3_summary_table.csv")

print("\n--- Suggested dissertation text (Findings, RQ3) ---")
for row in summary_rows:
    gv = row["grouping_variable"]
    print(
        f"\n'Predicted {TARGET_1_LABEL.lower()} was highest among {row[f'highest_{TARGET_1}']} "
        f"({gv.replace('_', ' ')}) at {row[f'highest_{TARGET_1}_value']:.2f}%, and lowest among "
        f"{row[f'lowest_{TARGET_1}']} at {row[f'lowest_{TARGET_1}_value']:.2f}%. Predicted "
        f"{TARGET_2_LABEL.lower()} was highest among {row[f'highest_{TARGET_2}']} at "
        f"{row[f'highest_{TARGET_2}_value']:.2f}%, and lowest among {row[f'lowest_{TARGET_2}']} "
        f"at {row[f'lowest_{TARGET_2}_value']:.2f}%.'"
    )

# ═══════════════════════════════════════════════════════════════════════════
# DONE
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("RQ3 LIGHTWEIGHT SUBGROUP ANALYSIS COMPLETE")
print("=" * 80)
print(f"""
Files generated in {OUT_DIR}/:
  rq3_predictions_with_subgroups.csv   — full predictions + subgroup labels, all companies
  rq3_grouped_means_by_industry.csv    — grouped stats by industry
  rq3_grouped_means_by_company_size.csv — grouped stats by company size
  rq3_grouped_means_by_region.csv      — grouped stats by region
  rq3_chart_industry.png               — grouped bar chart, by industry
  rq3_chart_company_size.png           — grouped bar chart, by company size
  rq3_chart_region.png                 — grouped bar chart, by region
  rq3_summary_table.csv                — highest/lowest category per grouping variable

No new models were trained and no new SHAP analysis was run — this reuses
the XGBoost models already trained for RQ1/RQ2, consistent with the
lightweight (Option 1) scope decision for RQ3.
""")